In [ ]:
!pip install transformers torch torchvision scipy pandas

In [ ]:
import cv2
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

# ================= CONFIGURATION =================
INPUT_VIDEOS = [
    "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test.mp4",
    "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test2.mp4",
    "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test3.mp4",
]
OUTPUT_VIDEOS = [
    "/kaggle/working/CLIPSeg_Video_Output_1.mp4",
    "/kaggle/working/CLIPSeg_Video_Output_2.mp4",
    "/kaggle/working/CLIPSeg_Video_Output_3.mp4",
]

MODEL_ID = "CIDAS/clipseg-rd64-refined"
TEXT_PROMPT = "rat"
MASK_THRESHOLD = 0.4

device = "cuda" if torch.cuda.is_available() else "cpu"
# =================================================

def apply_overlay(image, mask, color=(255, 0, 255), alpha=0.5):
    """Blends a solid color over the masked region (Purple for CLIPSeg)."""
    overlay = np.full_like(image, color)
    blended = cv2.addWeighted(image, 1 - alpha, overlay, alpha, 0)
    res = image.copy()
    res[mask == 1] = blended[mask == 1]
    return res

def process_video(input_video, output_video, model, processor):
    cap = cv2.VideoCapture(input_video)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(5)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (w, h))

    print(f"--- STARTING CLIPSeg VIDEO INFERENCE: {input_video} ---")
    pbar = tqdm(total=total_frames)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(image_rgb)

        inputs = processor(text=[TEXT_PROMPT], images=[pil_image], padding="max_length", return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        if logits.dim() == 3:
            logits = logits.unsqueeze(1)
        elif logits.dim() == 2:
            logits = logits.unsqueeze(0).unsqueeze(0)

        upscaled_logits = torch.nn.functional.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
        probs = torch.sigmoid(upscaled_logits).squeeze().cpu().numpy()

        pred_mask = (probs > MASK_THRESHOLD).astype(np.uint8)
        res_frame = apply_overlay(frame, pred_mask)
        cv2.putText(res_frame, f"CLIPSeg Prompt: '{TEXT_PROMPT}'", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        out.write(res_frame)
        pbar.update(1)

    cap.release()
    out.release()
    pbar.close()
    print(f"--- DONE. SAVED TO {output_video} ---")

def main():
    print("--- LOADING CLIPSeg VLM ---")
    processor = CLIPSegProcessor.from_pretrained(MODEL_ID)
    model = CLIPSegForImageSegmentation.from_pretrained(MODEL_ID).to(device).eval()

    for input_video, output_video in zip(INPUT_VIDEOS, OUTPUT_VIDEOS):
        process_video(input_video, output_video, model, processor)

if __name__ == "__main__":
    main()
